In [1]:
!pip install -q -U transformers accelerate requests

import os
import json
import glob
import time
import requests
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.stats import spearmanr
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 38.4 MB/s eta 0:00:00
device: cuda


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p_ in model.parameters():
    p_.requires_grad_(False)

ln_f = model.model.language_model.norm
lm_head = model.lm_head
layers = model.model.language_model.layers

n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = layers[0].self_attn.o_proj.in_features // n_head
IMAGE_TOKEN_ID = model.config.image_token_id
assert len(layers) == n_layer
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim,", model.config.text_config.num_key_value_heads, "kv heads, image token id", IMAGE_TOKEN_ID)

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

28 language model layers, 28 heads, 128 head dim, 4 kv heads, image token id 151655


In [3]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_CACHE_DIR = "/kaggle/temp/vg"
os.makedirs(VG_CACHE_DIR, exist_ok=True)
VG_URLS = ["https://cs.stanford.edu/people/rak248/VG_100K", "https://cs.stanford.edu/people/rak248/VG_100K_2"]

NOISE_CANDIDATES = [1.0, 1.5, 2.0, 3.0, 4.0, 6.0]
TAIL_N = 5
GRAD_SCALE_CANDIDATES = [32.0, 8.0, 1.0]
GRAD_SCALE = GRAD_SCALE_CANDIDATES[0]
RECALL_K = 20
HEADROOM_MAX = 20000.0
GROUPS = ["visual", "text", "tail"]
N_ACC_PER_COMBO = 6
N_CAL = 10
N_HEAD_TOP = 30
N_HEAD_RAND = 30
N_SEED_Q = 4
MIN_TE = 1.0
COMBOS = [("attribute", "amber"), ("relation", "amber"), ("relation", "reefknot")]


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


def find_vg_image_path(image_id):
    local = os.path.join(VG_CACHE_DIR, f"{image_id}.jpg")
    if os.path.exists(local):
        return local
    for base in VG_URLS:
        try:
            r = requests.get(f"{base}/{image_id}.jpg", timeout=30)
        except requests.RequestException:
            continue
        if r.status_code == 200 and len(r.content) > 1000:
            with open(local, "wb") as f:
                f.write(r.content)
            return local
    return None


def resolve_image_path(q):
    qid = q["question_id"]
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    if qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None


with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)
assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}

with open(f"{INPUT}/dla-qwen/dla_meta.json") as f:
    dla_meta = json.load(f)
PAIRS = {k: tuple(v) for k, v in dla_meta["pairs"].items()}
for prefix, (yes_id, no_id) in PAIRS.items():
    words = [processor.tokenizer.decode([i]).strip() for i in (yes_id, no_id)]
    print(prefix, yes_id, no_id, words)
    assert [w.lower() for w in words] == ["yes", "no"], words
assert set(PAIRS) == {"amber_", "reefknot_"}, set(PAIRS)

with open(f"{INPUT}/atp-qwen-selection/selection.json") as f:
    selection = json.load(f)
all_sel = [qid for t in selection["selected"] for qid in selection["selected"][t]]
print(len(all_sel), "selected questions")

rng = np.random.default_rng(0)


def pick(stratum, n, combos):
    pool = [
        q for q in all_sel
        if selection["stratum"][q] == stratum
        and (question_by_id[q]["hallucination_type"], selection["source"][q]) in combos
    ]
    order = [str(x) for x in rng.permutation(pool)]
    out = []
    for q in order:
        path = resolve_image_path(question_by_id[q])
        if path is not None and os.path.exists(path):
            out.append(q)
        if len(out) == n:
            break
    return out


acc_set = []
for combo in COMBOS:
    acc_set += pick("true_yes", N_ACC_PER_COMBO, [combo]) + pick("true_no", N_ACC_PER_COMBO, [combo])
cal_set = (
    pick("false_yes", 4, [("attribute", "amber")]) + pick("false_yes", 2, [("relation", "amber")]) + pick("false_yes", 2, [("relation", "reefknot")])
    + pick("true_yes", 1, [("attribute", "amber")]) + pick("true_no", 1, [("relation", "reefknot")])
)[:N_CAL]
seed_set = cal_set[:N_SEED_Q]
all_check = list(dict.fromkeys(acc_set + cal_set))
print(len(acc_set), "accuracy questions,", len(cal_set), "calibration questions")

dla_ld = {}
for path in glob.glob(f"{INPUT}/dla-qwen/*_batch_*.pt"):
    for e in torch.load(path):
        if e["question_id"] in all_check:
            dla_ld[e["question_id"]] = float(e["logit_diff"])
assert set(dla_ld) == set(all_check), "dla-qwen is missing some checked questions"


def prep(qid):
    q = question_by_id[qid]
    inputs = build_inputs(load_image(resolve_image_path(q)), q["question_text"])
    ids = inputs["input_ids"]
    T = ids.shape[1]
    vis = ids[0] == IMAGE_TOKEN_ID
    tail = torch.zeros(T, dtype=torch.bool, device=ids.device)
    tail[-TAIL_N:] = True
    text = ~vis & ~tail
    assert int(vis.sum()) == int(inputs["image_grid_thw"].prod()) // 4 and not bool((vis & tail).any())
    yes_id, no_id = PAIRS[qid.split("_")[0] + "_"]
    return {
        "qid": qid, "T": T, "inputs": inputs, "vis": vis, "yes_id": yes_id, "no_id": no_id,
        "masks": {"visual": vis.float().cpu(), "text": text.float().cpu(), "tail": tail.float().cpu()},
        "sign": 1.0 if q["ground_truth_answer"].strip().lower() == "yes" else -1.0,
    }


prepped = {qid: prep(qid) for qid in all_check}
p0 = prepped[cal_set[0]]
print("prompt tail:", processor.tokenizer.convert_ids_to_tokens(p0["inputs"]["input_ids"][0, -TAIL_N - 3:].tolist()))
print("positions:", {g: int(p0["masks"][g].sum()) for g in GROUPS}, "of", p0["T"])
print("sequence length over checked questions: min", min(p["T"] for p in prepped.values()), "max", max(p["T"] for p in prepped.values()))

amber_ 9454 2753 ['Yes', 'No']
reefknot_ 9693 2152 ['yes', 'no']
2927 selected questions
36 accuracy questions, 10 calibration questions


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


prompt tail: ['Ġthis', 'Ġimage', '?', '<|im_end|>', 'Ċ', '<|im_start|>', 'assistant', 'Ċ']
positions: {'visual': 988, 'text': 24, 'tail': 5} of 1017
sequence length over checked questions: min 296 max 1045


In [4]:
STATE = {"noise": None, "seed": 0, "capture": False, "grad": False, "live": None, "vis": None, "patch": None, "final_h": None, "probe": None}


def _layer_pre(l):
    def hook(module, args, kwargs):
        h = args[0] if args else kwargs["hidden_states"]
        changed = False
        if l == 0:
            if STATE["noise"] is not None:
                vis = STATE["vis"]
                h = h.clone()
                x = h[:, vis].float()
                generator = torch.Generator().manual_seed(STATE["seed"])
                noise = torch.randn(x.shape, generator=generator).to(h.device)
                noisy = x + noise * x.std() * STATE["noise"]
                h[:, vis] = noisy.to(h.dtype)
                changed = True
                if STATE["probe"] is not None:
                    STATE["probe"]["embed"] = (float(x.std()), float(x.abs().max()), float(noisy.abs().max()), float(h.float().abs().max()))
            if STATE["grad"]:
                h = h.detach().requires_grad_(True)
                changed = True
        if STATE["probe"] is not None:
            STATE["probe"].setdefault("layers", []).append(float(h.float().abs().max()))
        if STATE["capture"]:
            STATE["live"]["rp"][l] = h
        if not changed:
            return None
        if args:
            return (h,) + tuple(args[1:]), kwargs
        kwargs = dict(kwargs)
        kwargs["hidden_states"] = h
        return args, kwargs
    return hook


def _z_pre(l):
    def hook(module, args):
        z = args[0]
        patch = STATE["patch"]
        if patch is not None and patch["kind"] == "head" and patch["layer"] == l:
            T = z.shape[1]
            z4 = z.clone().view(1, T, n_head, head_dim)
            src = patch["src"].view(T, n_head, head_dim)
            idx = patch["idx"].to(z.device)
            z4[:, idx, patch["head"], :] = src[idx.to(src.device), patch["head"], :].to(z.device, z.dtype)
            z = z4.view(1, T, n_head * head_dim)
        if STATE["capture"]:
            STATE["live"]["z"][l] = z
        return (z,) + tuple(args[1:])
    return hook


def _mlp_fwd(l):
    def hook(module, args, output):
        out = output
        patch = STATE["patch"]
        if patch is not None and patch["kind"] == "mlp" and patch["layer"] == l:
            out = out.clone()
            idx = patch["idx"].to(out.device)
            out[:, idx] = patch["src"][idx.to(patch["src"].device)].to(out.device, out.dtype).unsqueeze(0)
        if STATE["capture"]:
            STATE["live"]["mlp"][l] = out
        return out
    return hook


def _final_norm_pre(module, args):
    if STATE["capture"]:
        STATE["final_h"] = args[0][0, -1].detach().float()


_handles = []
for l, layer in enumerate(layers):
    _handles.append(layer.register_forward_pre_hook(_layer_pre(l), with_kwargs=True))
    _handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_z_pre(l)))
    _handles.append(layer.mlp.register_forward_hook(_mlp_fwd(l)))
_handles.append(ln_f.register_forward_pre_hook(_final_norm_pre))
print(len(_handles), "hooks registered")

85 hooks registered


In [5]:
def mask_on(p, gname, dev):
    cache = p.setdefault("_mask", {})
    key = (gname, str(dev))
    if key not in cache:
        cache[key] = p["masks"][gname].to(dev)
    return cache[key]


def group_idx(p, gname):
    if gname == "all":
        return torch.arange(p["T"])
    return torch.nonzero(p["masks"][gname] > 0)[:, 0]


def make_patch(kind, layer, idx, src, head=None):
    return {"kind": kind, "layer": layer, "idx": idx, "src": src, "head": head}


def read_delta(p, out):
    logits = out.logits[0, -1].float()
    return p["sign"] * (logits[p["yes_id"]] - logits[p["no_id"]])


def capture_run(p, noise, seed, backward):
    STATE.update(noise=noise, seed=seed, capture=True, grad=backward, vis=p["vis"], patch=None, final_h=None)
    STATE["live"] = {"rp": [None] * n_layer, "z": [None] * n_layer, "mlp": [None] * n_layer}
    grads, finite = None, True
    with torch.set_grad_enabled(backward):
        out = model(**p["inputs"], use_cache=False, logits_to_keep=1)
        delta = read_delta(p, out)
        live = STATE["live"]
        if backward:
            raw = torch.autograd.grad(delta * GRAD_SCALE, live["z"] + live["mlp"] + live["rp"])
            finite = all(bool(torch.isfinite(g).all()) for g in raw)
            grads = {"z": list(raw[:n_layer]), "mlp": list(raw[n_layer:2 * n_layer]), "rp": list(raw[2 * n_layer:])}
    acts = {k: [t.detach() for t in live[k]] for k in ("z", "mlp", "rp")}
    result = {"acts": acts, "grads": grads, "delta": float(delta.detach()), "final_h": STATE["final_h"], "finite": finite}
    STATE.update(capture=False, grad=False, live=None, noise=None, final_h=None)
    del out, delta, live
    return result


@torch.no_grad()
def delta_with(p, noise, seed, patch=None):
    STATE.update(noise=noise, seed=seed, capture=False, grad=False, vis=p["vis"], patch=patch)
    out = model(**p["inputs"], use_cache=False, logits_to_keep=1)
    value = float(read_delta(p, out))
    STATE.update(noise=None, patch=None)
    return value


def atp_effects(A, B, G, p):
    head = torch.zeros(len(GROUPS), n_layer, n_head)
    mlp = torch.zeros(len(GROUPS), n_layer)
    rp = torch.zeros(len(GROUPS), n_layer)
    T = p["T"]
    for l in range(n_layer):
        dz = ((A["z"][l].float() - B["z"][l].float()) * G["z"][l].float() / GRAD_SCALE)[0].view(T, n_head, head_dim).sum(-1)
        dm = ((A["mlp"][l].float() - B["mlp"][l].float()) * G["mlp"][l].float() / GRAD_SCALE)[0].sum(-1)
        dr = ((A["rp"][l].float() - B["rp"][l].float()) * G["rp"][l].float() / GRAD_SCALE)[0].sum(-1)
        for gi, g in enumerate(GROUPS):
            head[gi, l] = (mask_on(p, g, dz.device) @ dz).cpu()
            mlp[gi, l] = (mask_on(p, g, dm.device) @ dm).cpu()
            rp[gi, l] = (mask_on(p, g, dr.device) @ dr).cpu()
    return {"head": head, "mlp": mlp, "rp": rp}

In [6]:
t0 = time.time()
clean_delta = {qid: delta_with(prepped[qid], None, 0) for qid in all_check}
print(f"{len(all_check)} clean forwards in {time.time() - t0:.1f} s ({(time.time() - t0) / len(all_check):.2f} s each)")

err = {qid: abs(clean_delta[qid] * prepped[qid]["sign"] - dla_ld[qid]) for qid in all_check}
print(f"clean logit difference vs dla-qwen: max err {max(err.values()):.4f}, mean err {np.mean(list(err.values())):.4f}")
assert all(np.isfinite(list(clean_delta.values()))), "non-finite clean deltas"
assert max(err.values()) < 0.25, "prompt or readout differs from the DLA collection"

acc_combo = [(question_by_id[q]["hallucination_type"], selection["source"][q]) for q in acc_set]
clean_ok = np.array([clean_delta[q] > 0 for q in acc_set])
print(f"clean: accuracy {clean_ok.mean():.2f} on {len(acc_set)} balanced true_yes and true_no questions")
assert clean_ok.mean() > 0.85, "clean accuracy is too low"

noise_acc, noise_finite, noise_gap = {}, {}, {}
for mult in NOISE_CANDIDATES:
    deltas = np.array([delta_with(prepped[qid], mult, 0) for qid in acc_set])
    noise_finite[mult] = bool(np.isfinite(deltas).all())
    noise_acc[mult] = float(np.mean(deltas > 0)) if noise_finite[mult] else float("nan")
    fin = deltas[np.isfinite(deltas)]
    noise_gap[mult] = float(np.mean(fin)) if len(fin) else float("nan")
    by_combo = {f"{t[:3]}/{s[:3]}": round(float(np.mean(deltas[[i for i, c in enumerate(acc_combo) if c == (t, s)]] > 0)), 2) for t, s in COMBOS} if noise_finite[mult] else {}
    print(f"noise x{mult}: finite {noise_finite[mult]}, accuracy {noise_acc[mult]:.2f}, mean delta {noise_gap[mult]:+.2f}, non-finite runs {int((~np.isfinite(deltas)).sum())}/{len(deltas)}, accuracy by source {by_combo}")
ok = [m for m in NOISE_CANDIDATES if noise_finite[m] and 0.3 <= noise_acc[m] <= 0.7]
assert ok, "no finite near-chance noise level"


def last_layer_max(mult):
    vals = []
    for qid in acc_set:
        STATE["probe"] = {}
        delta_with(prepped[qid], mult, 0)
        vals.append(STATE["probe"]["layers"][-1])
        STATE["probe"] = None
    return float(np.max(vals))


NOISE_MULT = None
for m in sorted(ok, reverse=True):
    peak = last_layer_max(m)
    print(f"noise x{m}: max |residual| entering the last layer over {len(acc_set)} questions {peak:.0f} (fp16 limit 65504, required below {HEADROOM_MAX:.0f})")
    if peak < HEADROOM_MAX:
        NOISE_MULT = m
        break
assert NOISE_MULT is not None, "every near-chance noise level is too close to the fp16 limit"
print(f"using NOISE_MULT = {NOISE_MULT} (the largest multiplier that is finite, near chance and has fp16 headroom)")

p = prepped[cal_set[0]]
for scale in GRAD_SCALE_CANDIDATES:
    GRAD_SCALE = scale
    c = capture_run(p, None, 0, True)
    fin = c["finite"]
    del c
    torch.cuda.empty_cache()
    print(f"gradient scale {scale}: all gradients finite {fin}")
    if fin:
        break
assert fin, "gradients are not finite at any scale"
GRAD_SCALE_USED = GRAD_SCALE

46 clean forwards in 30.7 s (0.67 s each)
clean logit difference vs dla-qwen: max err 0.0156, mean err 0.0003
clean: accuracy 1.00 on 36 balanced true_yes and true_no questions
noise x1.0: finite True, accuracy 1.00, mean delta +6.29, non-finite runs 0/36, accuracy by source {'att/amb': 1.0, 'rel/amb': 1.0, 'rel/ree': 1.0}
noise x1.5: finite True, accuracy 0.94, mean delta +5.92, non-finite runs 0/36, accuracy by source {'att/amb': 1.0, 'rel/amb': 0.92, 'rel/ree': 0.92}
noise x2.0: finite True, accuracy 0.86, mean delta +4.90, non-finite runs 0/36, accuracy by source {'att/amb': 0.92, 'rel/amb': 0.75, 'rel/ree': 0.92}
noise x3.0: finite True, accuracy 0.64, mean delta +2.14, non-finite runs 0/36, accuracy by source {'att/amb': 0.67, 'rel/amb': 0.58, 'rel/ree': 0.67}
noise x4.0: finite True, accuracy 0.53, mean delta +0.33, non-finite runs 0/36, accuracy by source {'att/amb': 0.58, 'rel/amb': 0.5, 'rel/ree': 0.5}
noise x6.0: finite True, accuracy 0.50, mean delta +0.05, non-finite runs 

In [7]:
SEED0 = 1000
cal_results = []
timing = {"capture": [], "exact": []}
u_cache = {}


def u_for(p):
    key = (p["yes_id"], p["no_id"])
    if key not in u_cache:
        u_cache[key] = ln_f.weight.float() * (lm_head.weight[p["yes_id"]] - lm_head.weight[p["no_id"]]).float()
    return u_cache[key]


for qi, qid in enumerate(cal_set):
    p = prepped[qid]
    seed = SEED0 + qi
    t0 = time.time()
    c = capture_run(p, None, seed, True)
    r = capture_run(p, NOISE_MULT, seed, False)
    timing["capture"].append(time.time() - t0)
    assert c["finite"], f"non-finite gradients for {qid}: lower the gradient scale"
    assert np.isfinite(r["delta"]) and np.isfinite(c["delta"]), f"non-finite delta for {qid}"
    TE = c["delta"] - r["delta"]
    atp = atp_effects(r["acts"], c["acts"], c["grads"], p)
    selfcheck = abs(delta_with(p, NOISE_MULT, seed) - r["delta"])

    h = c["final_h"]
    rms = torch.sqrt(h.pow(2).mean() + ln_f.variance_epsilon)
    dm = (r["acts"]["mlp"][-1][0, -1].float() - c["acts"]["mlp"][-1][0, -1].float()).to(h.device)
    anchor = float(p["sign"] * (dm @ u_for(p).to(h.device)) / rms)

    head_all = atp["head"].sum(0).flatten()
    order = torch.argsort(head_all.abs(), descending=True).tolist()
    top = order[:N_HEAD_TOP]
    near = order[N_HEAD_TOP:N_HEAD_TOP + N_HEAD_RAND]
    exact = {"mlp": {}, "head": {}}
    t1 = time.time()
    n_fwd = 0
    for l in range(n_layer):
        for gname in ["all"] + GROUPS:
            patch = make_patch("mlp", l, group_idx(p, gname), r["acts"]["mlp"][l][0])
            exact["mlp"][(gname, l)] = delta_with(p, None, seed, patch) - c["delta"]
            n_fwd += 1
    for flat in top + near:
        l, h_ = divmod(int(flat), n_head)
        patch = make_patch("head", l, group_idx(p, "all"), r["acts"]["z"][l][0], h_)
        exact["head"][(l, h_)] = delta_with(p, None, seed, patch) - c["delta"]
        n_fwd += 1
    timing["exact"].append((time.time() - t1) / n_fwd)

    cal_results.append({
        "qid": qid, "stratum": selection["stratum"][qid], "TE": TE, "delta_clean": c["delta"], "delta_corrupt": r["delta"],
        "selfcheck": selfcheck, "anchor": anchor, "atp": atp, "exact": exact, "near": set(near), "T": p["T"],
    })
    print(f"[{qi + 1}/{len(cal_set)}] {qid} {selection['stratum'][qid]} T={p['T']}: clean {c['delta']:+.2f} corrupt {r['delta']:+.2f} TE {TE:+.2f} | "
          f"grad run {timing['capture'][-1]:.1f} s, exact {timing['exact'][-1]:.2f} s/forward")
    del c, r, atp
    torch.cuda.empty_cache()

[1/10] amber_2332 false_yes T=1017: clean -7.50 corrupt +7.68 TE -15.18 | grad run 4.5 s, exact 1.59 s/forward
[2/10] amber_4470 false_yes T=421: clean -0.72 corrupt +11.41 TE -12.12 | grad run 1.8 s, exact 0.65 s/forward
[3/10] amber_3356 false_yes T=420: clean -1.80 corrupt +6.56 TE -8.36 | grad run 1.8 s, exact 0.65 s/forward
[4/10] amber_6824 false_yes T=420: clean -5.00 corrupt +9.27 TE -14.27 | grad run 1.8 s, exact 0.65 s/forward
[5/10] amber_15035 false_yes T=1039: clean -1.70 corrupt +5.80 TE -7.50 | grad run 5.1 s, exact 1.84 s/forward
[6/10] amber_14502 false_yes T=330: clean -2.16 corrupt +8.94 TE -11.09 | grad run 1.4 s, exact 0.49 s/forward
[7/10] reefknot_2383351_1 false_yes T=360: clean -0.62 corrupt +3.66 TE -4.28 | grad run 1.4 s, exact 0.51 s/forward
[8/10] reefknot_2350156_1 false_yes T=303: clean -2.70 corrupt +5.70 TE -8.41 | grad run 1.4 s, exact 0.51 s/forward
[9/10] amber_5097 true_yes T=420: clean +2.23 corrupt -11.62 TE +13.86 | grad run 1.7 s, exact 0.65 s/f

In [8]:
def pair_frame():
    rows = []
    for res in cal_results:
        a, e = res["atp"], res["exact"]
        mlp_all, head_all = a["mlp"].sum(0), a["head"].sum(0)
        for (g, l), v in e["mlp"].items():
            atp_v = mlp_all[l].item() if g == "all" else a["mlp"][GROUPS.index(g), l].item()
            rows.append({"qid": res["qid"], "kind": f"mlp_{g}", "comp": l, "atp": atp_v, "exact": v, "pool": "mlp"})
        for (l, h_), v in e["head"].items():
            rows.append({"qid": res["qid"], "kind": "head_all", "comp": l * n_head + h_, "atp": head_all[l, h_].item(),
                         "exact": v, "pool": "near" if l * n_head + h_ in res["near"] else "top"})
    return pd.DataFrame(rows)


def calib_stats(df):
    no_text = df[df.kind != "mlp_text"]
    out = {
        "pooled_spearman": float(spearmanr(df.atp, df.exact)[0]),
        "pooled_spearman_no_text": float(spearmanr(no_text.atp, no_text.exact)[0]),
        "slope_exact_on_atp": float((df.atp * df.exact).sum() / (df.atp ** 2).sum()),
    }
    for kind, sub in df.groupby("kind"):
        out[f"spearman_{kind}"] = float(spearmanr(sub.atp, sub.exact)[0])
    key = df[df.kind.isin(["mlp_all", "head_all"])]
    rec10, recK, sgn, perq, miss = [], [], [], [], []
    for qid, sub in key.groupby("qid"):
        sub = sub.reset_index(drop=True)
        k1, k2 = min(10, len(sub)), min(RECALL_K, len(sub))
        rec10.append(len(set(sub.exact.abs().nlargest(k1).index) & set(sub.atp.abs().nlargest(k1).index)) / k1)
        recK.append(len(set(sub.exact.abs().nlargest(k2).index) & set(sub.atp.abs().nlargest(k2).index)) / k2)
        big = sub[sub.exact.abs() > 0.1 * sub.exact.abs().max()]
        sgn.append(float((np.sign(big.atp) == np.sign(big.exact)).mean()))
        perq.append(float(spearmanr(sub.atp, sub.exact)[0]))
        heads = sub[sub.kind == "head_all"]
        th, nh = heads[heads.pool == "top"], heads[heads.pool == "near"]
        if len(th) and len(nh):
            thr = th.exact.abs().nlargest(min(10, len(th))).iloc[-1]
            miss.append(float((nh.exact.abs() > thr).sum()))
    out.update(top10_recall=float(np.mean(rec10)), **{f"top{RECALL_K}_recall": float(np.mean(recK))},
               sign_agreement=float(np.mean(sgn)), per_question_spearman=float(np.mean(perq)),
               near_miss_heads=float(np.mean(miss)) if miss else float("nan"))
    return out


strong = [res["qid"] for res in cal_results if abs(res["TE"]) > MIN_TE]
df_pairs = pair_frame()
calib = {"all": calib_stats(df_pairs)}
print(f"--- noising, true gradient: {len(df_pairs)} (question, component) pairs, all {len(cal_results)} questions ---")
print({k: round(v, 3) for k, v in calib["all"].items()})
if len(strong) >= 2:
    calib["strong"] = calib_stats(df_pairs[df_pairs.qid.isin(strong)])
    print(f"    only the {len(strong)} questions with |TE| > {MIN_TE}:", {k: round(v, 3) for k, v in calib["strong"].items()})
s = calib["all"]
meets = bool(s["pooled_spearman_no_text"] >= 0.8 and s[f"top{RECALL_K}_recall"] >= 0.7 and s["near_miss_heads"] <= 2)
print(f"    meets the bars (pooled spearman without text >= 0.8, top-{RECALL_K} recall >= 0.7, near-miss heads <= 2): {meets}")

for qid, sub in df_pairs[df_pairs.kind.isin(["mlp_all", "head_all"])].groupby("qid"):
    print(f"{qid:24s} spearman {spearmanr(sub.atp, sub.exact)[0]:.3f} | largest exact effect {sub.exact.abs().max():.3f}")

mid = min(n_layer // 2 + 2, n_layer - 1)
print(f"--- L{mid} MLP, all positions: atp vs exact per question ---")
for res in cal_results:
    print(f"{res['qid']:24s} {res['stratum']:10s} TE {res['TE']:+7.2f} | atp {res['atp']['mlp'].sum(0)[mid]:+7.3f} exact {res['exact']['mlp'][('all', mid)]:+7.3f}")

print("--- residual stream identity: atp of resid_pre at all positions / -TE; exact is 1 ---")
layers_rp = sorted({0, n_layer // 4, n_layer // 2, 3 * n_layer // 4, n_layer - 1})
use = [res for res in cal_results if abs(res["TE"]) > MIN_TE]
for l in layers_rp:
    ratio = np.median([res["atp"]["rp"].sum(0)[l].item() / (-res["TE"]) for res in use]) if use else float("nan")
    print(f"resid_pre L{l:2d}: {ratio:.2f}  ({len(use)} questions with |TE| > {MIN_TE})")

print("--- anchor: last-layer MLP at all positions ---")
for res in cal_results:
    print(f"{res['qid']:24s} atp {res['atp']['mlp'].sum(0)[-1]:+7.3f} dla {res['anchor']:+7.3f} exact {res['exact']['mlp'][('all', n_layer - 1)]:+7.3f}")
print(f"grad run vs no-grad run delta, max abs diff: {max(r_['selfcheck'] for r_ in cal_results):.4f}")

--- noising, true gradient: 1720 (question, component) pairs, all 10 questions ---
{'pooled_spearman': 0.737, 'pooled_spearman_no_text': 0.751, 'slope_exact_on_atp': 0.667, 'spearman_head_all': 0.973, 'spearman_mlp_all': 0.62, 'spearman_mlp_tail': 0.796, 'spearman_mlp_text': 0.622, 'spearman_mlp_visual': 0.36, 'top10_recall': 0.59, 'top20_recall': 0.705, 'sign_agreement': 0.907, 'per_question_spearman': 0.779, 'near_miss_heads': 0.4}
    only the 10 questions with |TE| > 1.0: {'pooled_spearman': 0.737, 'pooled_spearman_no_text': 0.751, 'slope_exact_on_atp': 0.667, 'spearman_head_all': 0.973, 'spearman_mlp_all': 0.62, 'spearman_mlp_tail': 0.796, 'spearman_mlp_text': 0.622, 'spearman_mlp_visual': 0.36, 'top10_recall': 0.59, 'top20_recall': 0.705, 'sign_agreement': 0.907, 'per_question_spearman': 0.779, 'near_miss_heads': 0.4}
    meets the bars (pooled spearman without text >= 0.8, top-20 recall >= 0.7, near-miss heads <= 2): False
amber_14502              spearman 0.814 | largest exact 

In [9]:
vec = lambda a: torch.cat([a["head"].sum(0).flatten(), a["mlp"].sum(0)]).numpy()

seed_stab, te_spread = [], []
for qid in seed_set:
    p = prepped[qid]
    c = capture_run(p, None, 0, True)
    runs, tes = [], []
    for s_ in range(3):
        r = capture_run(p, NOISE_MULT, 100 + s_, False)
        runs.append(vec(atp_effects(r["acts"], c["acts"], c["grads"], p)))
        tes.append(c["delta"] - r["delta"])
        del r
    seed_stab.append(np.mean([spearmanr(runs[i], runs[j])[0] for i in range(3) for j in range(i + 1, 3)]))
    te_spread.append(float(np.std(tes)))
    del c
    torch.cuda.empty_cache()
print(f"rank correlation of effects across noise seeds, mean {np.mean(seed_stab):.3f} (per question {np.round(seed_stab, 3).tolist()})")
print("std of TE across seeds per question:", np.round(te_spread, 3).tolist())

grad_scale_err = None
if GRAD_SCALE_USED != 1.0:
    p = prepped[seed_set[0]]
    r = capture_run(p, NOISE_MULT, 100, False)
    vecs = {}
    for scale in (GRAD_SCALE_USED, 1.0):
        GRAD_SCALE = scale
        c = capture_run(p, None, 0, True)
        vecs[scale] = vec(atp_effects(r["acts"], c["acts"], c["grads"], p)) if c["finite"] else None
        del c
        torch.cuda.empty_cache()
    GRAD_SCALE = GRAD_SCALE_USED
    if vecs[1.0] is not None and np.isfinite(vecs[1.0]).all():
        ref_ = vecs[GRAD_SCALE_USED]
        grad_scale_err = float(np.max(np.abs(vecs[1.0] - ref_)) / np.max(np.abs(ref_)))
        print(f"grad scale 1 vs {GRAD_SCALE_USED}: max difference as a fraction of the largest effect {grad_scale_err:.4f}")
    else:
        print("grad scale 1 is not finite or underflows; the larger scale is required")

rank correlation of effects across noise seeds, mean 0.885 (per question [0.903, 0.876, 0.873, 0.889])
std of TE across seeds per question: [0.133, 0.635, 0.876, 0.521]
grad scale 1 vs 32.0: max difference as a fraction of the largest effect 0.0018


In [10]:
summary = {
    "noise_accuracy": {str(k): v for k, v in noise_acc.items()},
    "noise_finite": {str(k): v for k, v in noise_finite.items()},
    "noise_mean_delta": {str(k): v for k, v in noise_gap.items()},
    "noise_mult": NOISE_MULT, "tail_n": TAIL_N, "grad_scale": GRAD_SCALE_USED,
    "clean_vs_dla_max_err": float(max(err.values())),
    "calibration": calib,
    "seed_stability": float(np.mean(seed_stab)),
    "grad_scale_diff_fraction_of_max": grad_scale_err,
    "sec_per_grad_run": float(np.mean(timing["capture"])),
    "sec_per_exact_forward": float(np.mean(timing["exact"])),
    "seq_len_min_max": [min(p["T"] for p in prepped.values()), max(p["T"] for p in prepped.values())],
    "max_gpu_mem_gb": [torch.cuda.max_memory_allocated(i) / 1e9 for i in range(torch.cuda.device_count())],
}
for key, val in summary.items():
    print(key, val)

sec = summary["sec_per_grad_run"] + summary["sec_per_exact_forward"]
print(f"noising only (one backward on the clean run, one forward on the corrupt run): about {sec:.1f} s per question, "
      f"{len(all_sel) * sec / 3600:.1f} h for all {len(all_sel)} selected questions")

with open("/kaggle/working/atp_validate_qwen.json", "w") as f:
    json.dump(summary, f, indent=1)

noise_accuracy {'1.0': 1.0, '1.5': 0.9444444444444444, '2.0': 0.8611111111111112, '3.0': 0.6388888888888888, '4.0': 0.5277777777777778, '6.0': 0.5}
noise_finite {'1.0': True, '1.5': True, '2.0': True, '3.0': True, '4.0': True, '6.0': True}
noise_mean_delta {'1.0': 6.287326388888889, '1.5': 5.915798611111111, '2.0': 4.898003472222222, '3.0': 2.1371527777777777, '4.0': 0.3259548611111111, '6.0': 0.0546875}
noise_mult 6.0
tail_n 5
grad_scale 32.0
clean_vs_dla_max_err 0.015625
calibration {'all': {'pooled_spearman': 0.7371518627312055, 'pooled_spearman_no_text': 0.7514114116849536, 'slope_exact_on_atp': 0.6672688022397757, 'spearman_head_all': 0.9731132221643671, 'spearman_mlp_all': 0.6202942197503006, 'spearman_mlp_tail': 0.7961330942068614, 'spearman_mlp_text': 0.6215088183039466, 'spearman_mlp_visual': 0.36042447955828777, 'top10_recall': 0.59, 'top20_recall': 0.7050000000000001, 'sign_agreement': 0.9069821065307119, 'per_question_spearman': 0.7786349527800512, 'near_miss_heads': 0.4}, 